In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os

In [2]:
def get_range(output, target, n_step=-1):
    if n_step > 0:
        output = output[:, :n_step]
        target = target[:, 10:n_step]
    else:
        target = target[:, 10:]

    diff = target - output # (b, t, x, y, c)

    max1 = np.maximum(np.max(target, axis=(-4, -3, -2)), np.max(output, axis=(-4, -3, -2)))
    min1 = np.minimum(np.min(target, axis=(-4, -3, -2)), np.min(output, axis=(-4, -3, -2)))

    max2 = np.max(diff, axis=(-4, -3, -2))
    min2 = np.min(diff, axis=(-4, -3, -2))

    return np.stack([max1, max2], axis=1), np.stack([min1, min2], axis=1)

In [3]:
def plot_ax_formal(ax, array, side_title=None, title=None, fontsize=15, cmap="RdBu_r", sym=False, ran_max=None, ran_min=None, pos=-1):
    if sym:
        if ran_max is None:
            vmax = np.max(np.abs(array))
        else:
            vmax = max(ran_max, ran_min, -ran_min)
        im = ax.imshow(array, cmap=cmap, vmin=-vmax, vmax=vmax)
    else:
        if ran_max is None:
            im = ax.imshow(array, cmap=cmap)
        else:
            im = ax.imshow(array, cmap=cmap, vmin=ran_min, vmax=ran_max)
    
    if pos < 0 or pos == 2:
        cbar = plt.colorbar(im, ax=ax, fraction=0.046)
        cbar.ax.tick_params(labelsize=fontsize)
    
    if pos == 0:
        ax.annotate(side_title, (-0.1, 0.5), xycoords = 'axes fraction', rotation = 90, va = 'center', fontsize = fontsize+8)

    ax.tick_params(left=False, right=False, labelleft=False, labelbottom=False, bottom=False)
    if title is not None:
        ax.set_title(title, fontsize=fontsize)

def plot_2d(
    output: np.ndarray,
    data_all,
    plot_title="",
    input_len=10,
    output_plot_len=-1,
    dim=-1,
    save_path="",
    ran_max=None,
    ran_min=None,
):
    """
    output:     (output_len, x_num, x_num, data_dim)
    data_all:   (input_len + output_len, x_num, x_num, data_dim)
    """

    if dim < 0:
        dim = output.shape[-1]
    

    if output_plot_len < 0:
        output_len = output.shape[0]
        output_plot_len = output_len
    else:
        output_len = output_plot_len

    total_col = output_plot_len
    total_row = dim * 3
    fig, axs = plt.subplots(total_row, total_col, squeeze=False, figsize=(4 * total_col, 4 * total_row), layout='constrained')

    for idx, output_step in enumerate(range(output_len - output_plot_len, output_len)):
        step = input_len + output_step
        if output_step + 1 == output_len:
            pos = 2
        elif output_step == 0:
            pos = 0
        else:
            pos = 1
                    
        for j in range(dim):
            cur_target = data_all[step, ..., j]
            cur_output = output[output_step, ..., j]
            diff = cur_target - cur_output
            # diff = np.abs(cur_target - cur_output)

            if ran_max is None:
                plot_ax_formal(axs[j * 3 + 1, idx], cur_output, title="Prediction")
                plot_ax_formal(axs[j * 3, idx], cur_target, title=f"Target Step {output_step+1}")
                plot_ax_formal(axs[j * 3 + 2, idx], diff, title="Difference", sym=True)
            else:
                plot_ax_formal(axs[j * 3 + 1, idx], cur_output, "Prediction", ran_max=ran_max[0][j], ran_min=ran_min[0][j], pos=pos)
                plot_ax_formal(axs[j * 3, idx], cur_target, "Target", f"Step {output_step+1}", ran_max=ran_max[0][j], ran_min=ran_min[0][j], pos=pos)
                plot_ax_formal(axs[j * 3 + 2, idx], diff, "Difference", sym=True, ran_max=ran_max[1][j], ran_min=ran_min[1][j], pos=pos)

    for ax in axs.flat:
        ax.label_outer()
        ax.tick_params(axis="both", labelsize=8)

    plt.suptitle(plot_title + "\n\n", fontsize=20)
    # plt.tight_layout()

    if save_path:
        plt.savefig(save_path)
        plt.close(fig)
    else:
        plt.show()

In [4]:
exp2name = {
    "1to1_auto_41": "BCAT",
    "fno_test_1": "FNO",
    "unet_1": "UNet",
    "vit_1": "ViT",
    "deeponet": "DeepONet",
    "mpp-b": "MPP-B",
    "mpp-l": "MPP-L",
    "dpot-m": "DPOT-M",
    "dpot-l": "DPOT-L",
}

def plot_compare(
    target,
    all_data,
    plot_title="",
    output_len=-1,
    save_path="",
    i = 0,
):

    dim = target.shape[-1]
    n_exp = len(all_data) + 1

    if output_len < 0:
        output_len = target.shape[1]

    total_col = output_len
    total_row = dim * n_exp
    fig, axs = plt.subplots(total_row, total_col, squeeze=False, figsize=(4 * total_col, 4 * total_row), layout='constrained')


    for idx in range(output_len):
        if idx + 1 == output_len:
            pos = 2
        elif idx == 0:
            pos = 0
        else:
            pos = 1
                    
        for j in range(dim):
            plot_ax_formal(axs[j * n_exp, idx], target[i, idx, ..., j], "Target", f"Step {idx+1}", pos=pos)

            for k, (exp, d) in enumerate(all_data.items()):
                diff = d["diff"][i, idx, ..., j]
                title = "{} ({:.2%})".format(exp2name[exp], d["error"][i])
                plot_ax_formal(axs[j * n_exp + k + 1, idx], diff, title, sym=True, pos=pos)

    for ax in axs.flat:
        ax.label_outer()
        ax.tick_params(axis="both", labelsize=8)

    plt.suptitle(plot_title + "\n\n", fontsize=20)

    if save_path:
        plt.savefig(save_path)
        plt.close(fig)
    else:
        plt.show()

Test

In [ ]:
expid = "1to1_auto_41"
exp = "cfdbench"
n_plot = 20
n_step = 6

path = f"/home/yuxuan/code/prose/fluids/src/checkpoint/save/{expid}/evals_all/outputs/{exp}.npz"
d = np.load(path)
output = d["output"][:, :n_step]
target = d["target"][:, :10 + n_step]
errors = d["errors"]
ran_max, ran_min = get_range(output, target)

i = 0
# plot_2d(output[i], target[i])
plot_2d(output[i], target[i], ran_max=ran_max[i], ran_min=ran_min[i], plot_title=f"{exp} | Idx {i} | {errors[i]:.4f}")

Plot all for BCAT

In [108]:
expid = "1to1_auto_41"
datasets = ["cfdbench", "com_ns", "incom_ns_arena_u", "incom_ns_arena", "incom_ns", "shallow_water"]

n_plot = 20
n_step = 6

for exp in datasets:
    path = f"/home/yuxuan/code/prose/fluids/src/checkpoint/save/{expid}/evals_all/outputs/{exp}.npz"
    d = np.load(path)
    output = d["output"][:, :n_step]
    target = d["target"][:, :10 + n_step]
    errors = d["errors"]
    ran_max, ran_min = get_range(output, target)
    
    save_folder = f"/home/yuxuan/code/prose/fluids/src/checkpoint/all_graphs/bcat_graph/{expid}/{exp}"
    os.makedirs(save_folder, exist_ok=True)

    
    for i in range(min(n_plot, errors.shape[0])):
        plot_2d(output[i], target[i], ran_max=ran_max[i], ran_min=ran_min[i], plot_title=f"{exp} | Idx {i} | {errors[i]:.4f}", save_path=f"{save_folder}/{exp}_{i}_{errors[i]:.3f}.png")


Plot for other models

In [ ]:
exp = "incom_ns_arena"
expids = ["fno_test_1", "deeponet", "unet_1", "vit_1", "mpp-l"]

n_plot = 20
n_step = 6

for expid in expids:
    path = f"/home/yuxuan/code/prose/fluids/src/checkpoint/save/{expid}/evals_all/outputs/{exp}.npz"
    d = np.load(path)
    output = d["output"][:, :n_step]
    target = d["target"][:, :10 + n_step]
    errors = d["errors"]
    ran_max, ran_min = get_range(output, target)
    
    save_folder = f"/home/yuxuan/code/prose/fluids/src/checkpoint/all_graphs/bcat_graph/{expid}/{exp}"
    os.makedirs(save_folder, exist_ok=True)

    
    for i in range(min(n_plot, errors.shape[0])):
        plot_2d(output[i], target[i], ran_max=ran_max[i], ran_min=ran_min[i], plot_title=f"{exp} | Idx {i} | {errors[i]:.4f}", save_path=f"{save_folder}/{exp}_{i}_{errors[i]:.3f}.png")


In [5]:
datasets = ["com_ns", "incom_ns_arena_u", "incom_ns_arena", "shallow_water"]
exp = "com_ns"
expids = ["fno_test_1", "deeponet", "unet_1", "vit_1", "mpp-l", "dpot-l"]
all_exps = expids + ["1to1_auto_41"]

n_plot = 20
n_step = 6

all_data = {}

for expid in all_exps:
    path = f"/home/yuxuan/code/prose/fluids/src/checkpoint/save/{expid}/evals_all/outputs/{exp}.npz"
    d = np.load(path)
    output = d["output"][:, :n_step]
    target = d["target"][:, 10:10 + n_step]
    errors = d["errors"]

    if expid.startswith("dpot") and "arena" in exp:
        target = target[...,[1,2,0]]
        output = output[...,[1,2,0]]

    all_data[expid] = {
        "exp": expid,
        "diff": target - output,
        "error": errors,
    }

# plot_compare(target, all_data)

In [9]:
save_folder = f"/home/yuxuan/code/prose/fluids/src/checkpoint/all_graphs/bcat_graph/compare/{exp}"
os.makedirs(save_folder, exist_ok=True)

for i in range(min(n_plot, target.shape[0])):
# for i in range(40, target.shape[0]):
    plot_compare(target, all_data, plot_title=f"{exp} | Idx {i}", save_path=f"{save_folder}/{exp}_{i}.png", i=i)

In [6]:
print(all_data["1to1_auto_41"]["error"])

[0.00057289 0.00038223 0.00056263 0.00052774 0.00047183 0.00066883
 0.00362337 0.00081832 0.00065481 0.00166579 0.00041455 0.0008042
 0.00168717 0.00068744 0.00099643 0.00105664 0.00163983 0.00093899
 0.00051141 0.00063566 0.00043348 0.00074547 0.00071108 0.00087262
 0.00130788 0.00053    0.00030481 0.00243561 0.00041064 0.00045551
 0.00091522 0.0063418  0.00087095 0.00058697 0.00143903 0.00259445
 0.00154216 0.00099859 0.00075507 0.00077683 0.00103425 0.00119913
 0.00063263 0.00130001 0.00028973 0.00188229 0.00206617 0.00687626
 0.00176408 0.00142742]


In [7]:
max(all_data["1to1_auto_41"]["error"])

0.006876261439174414

In [8]:
i = 17

for expid, d in all_data.items():
    print(expid, d["error"][i])

fno_test_1 0.037901461124420166
deeponet 0.042594488710165024
unet_1 0.014930968172848225
vit_1 0.0038809645920991898
mpp-l 0.005291629582643509
dpot-l 0.0013893404975533485
1to1_auto_41 0.0009389864280819893


In [40]:
# datasets = ["cfdbench", "com_ns", "incom_ns_arena_u", "incom_ns_arena", "incom_ns", "shallow_water"]
datasets = ["com_ns", "incom_ns_arena_u", "incom_ns_arena", "shallow_water"]

exp = "incom_ns_arena_u"

bcat_path = f"/home/yuxuan/code/prose/fluids/src/checkpoint/save/{expid}/evals_all/outputs/{exp}.npz"
# mpp_path = f"/home/yuxuan/code/prose/fluids/src/checkpoint/save/{expid}/evals_all/outputs/{exp}.npz"
mpp_path = f"/home/yuxuan/code/prose/fluids/src/checkpoint/save/mpp-l/evals_all/outputs/{exp}.npz"
dpot_path = f"/home/yuxuan/code/prose/fluids/src/checkpoint/save/dpot-l/evals_all/outputs/{exp}.npz"

mpp = np.load(mpp_path)["target"]
bcat = np.load(bcat_path)["target"]
dpot = np.load(dpot_path)["target"][...,[1,2,0]]

diff1 = mpp - bcat
print(np.sum(np.abs(diff1)))

diff2 = dpot - bcat
print(np.sum(np.abs(diff2)))

0.0
0.0
